# 🌽 Maize Leaf Binary Classifier — Universal Colab Notebook
### University of Eswatini | BSc Computer Science & Mathematics
---
**Healthy vs. Diseased | MobileNetV2 · Xception · InceptionV3 · VGG16 · ResNet50**

## ✅ How to use this notebook (READ FIRST)

1. **Enable GPU**: Click `Runtime → Change runtime type → Hardware accelerator → GPU (T4)` → Save
2. **Change `MODEL_NAME`** in **Cell 1** to the architecture assigned to *this* Google account
3. **Run all cells**: Click `Runtime → Run all` — then follow the prompts
4. **Results** (model + 4 plots + metrics) are automatically saved to your **Google Drive**

---
### Which model goes on which account?

| Account | Set `MODEL_NAME` to |
|---------|----------------------|
| Account 1 | `"MobileNetV2"` ← *primary — do this one first* |
| Account 2 | `"Xception"` |
| Account 3 | `"InceptionV3"` |
| Account 4 | `"VGG16"` |
| Account 5 | `"ResNet50"` |
| Account 6 | Spare (retry any that failed) |
| Account 7 | Spare (retry any that failed) |


## Cell 1 — ⚙️ Configuration
*Change only `MODEL_NAME`. Do not change anything else.*

In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║     ★  CHANGE ONLY THIS LINE — one per Google account  ★     ║
# ║                                                              ║
MODEL_NAME = "MobileNetV2"   # ← OPTIONS (copy exactly):        ║
#                                 "MobileNetV2"                  ║
#                                 "Xception"                     ║
#                                 "InceptionV3"                  ║
#                                 "VGG16"                        ║
#                                 "ResNet50"                     ║
# ╚══════════════════════════════════════════════════════════════╝

# ── Fixed hyperparameters (thesis specification) ─────────────────────────────
IMG_SIZE       = 224     # All architectures standardised to 224×224
BATCH_SIZE     = 32
PHASE1_EPOCHS  = 25      # Feature extraction phase (max 25, early-stop at 5)
PHASE2_EPOCHS  = 20      # ← Fine-tuning phase  ★ TIME-CAPPED AT 20 ★
PHASE1_LR      = 1e-3    # Adam lr — Phase 1
PHASE2_LR      = 1e-5    # Adam lr — Phase 2 (10× smaller to protect weights)
DROPOUT_RATE   = 0.50    # Dropout rate in classification head
RANDOM_SEED    = 42      # For reproducible splits

print(f"✅ Configuration set — Model: {MODEL_NAME}")
print(f"   Phase 1: max {PHASE1_EPOCHS} epochs | Phase 2: max {PHASE2_EPOCHS} epochs")


## Cell 2a — 📂 Mount Google Drive

**⚠️ If this cell fails with "mount failed":**
1. Look for a "Pop-ups blocked" icon in your Chrome address bar (top right)
2. Click it → select **"Always allow pop-ups from colab.research.google.com"**
3. Click the ▶ button on this cell again to re-run it
4. A sign-in pop-up will appear — click **Allow**

You should see: `Mounted at /content/drive` ✅

In [ ]:
# ── Mount Google Drive (your outputs will be saved here) ─────────────────────
# If this fails: allow pop-ups for colab.research.google.com in your browser,
# then click the ▶ play button on this cell again.
from google.colab import drive

try:
    drive.mount('/content/drive', force_remount=True)
    print("✅ Google Drive mounted successfully!")
except Exception as e:
    print(f"❌ Mount failed: {e}")
    print("\nFix: Allow pop-ups for colab.research.google.com in your browser")
    print("Then run this cell again (click the ▶ play button on the left).")


## Cell 2b — 📦 Install Packages & Imports

In [ ]:
# Install Kaggle API client
!pip install kaggle -q

import os, sys, json, shutil, glob, time, zipfile, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
warnings.filterwarnings('ignore')

# TensorFlow / Keras
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.applications import MobileNetV2, Xception, InceptionV3, VGG16, ResNet50
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau
from tensorflow.keras.preprocessing.image import ImageDataGenerator

# Scikit-learn
from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, roc_auc_score, roc_curve
)
from sklearn.utils.class_weight import compute_class_weight

# Create output folder on Drive
DRIVE_OUTPUT = f'/content/drive/MyDrive/MaizeClassifier/{MODEL_NAME}'
os.makedirs(DRIVE_OUTPUT,        exist_ok=True)
os.makedirs('/content/datasets', exist_ok=True)

# Fix random seeds for reproducibility
tf.random.set_seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

print(f"\n{'='*55}")
print(f"  TensorFlow : {tf.__version__}")
print(f"  GPU        : {tf.config.list_physical_devices('GPU')}")
print(f"  Model      : {MODEL_NAME}")
print(f"  Drive path : {DRIVE_OUTPUT}")
print(f"{'='*55}")


## Cell 3 — 🔑 Kaggle API Token Setup

**Kaggle changed their system** — they now show you the token on screen instead of giving a file.

1. Go back to the Kaggle API page where your token is shown (starts with `KGAT_...`)
2. Click the **copy icon** next to the token string to copy it
3. Paste it inside the quotes below, replacing the placeholder text
4. Run this cell

In [ ]:
import os, subprocess

# ─────────────────────────────────────────────────────────────────────
#  ★ PASTE YOUR KAGGLE API TOKEN INSIDE THE QUOTES BELOW ★
#  It starts with KGAT_ and looks like:  KGAT_abc123def456...
#  Copy it from your Kaggle Settings → API page.
# ─────────────────────────────────────────────────────────────────────
MY_KAGGLE_TOKEN = "PASTE_YOUR_TOKEN_HERE"

# ── Apply token (do not change anything below) ────────────────────────
os.environ['KAGGLE_API_TOKEN'] = MY_KAGGLE_TOKEN

os.makedirs('/root/.kaggle', exist_ok=True)
with open('/root/.kaggle/access_token', 'w') as f:
    f.write(MY_KAGGLE_TOKEN)
os.chmod('/root/.kaggle/access_token', 0o600)

# Test the connection
result = subprocess.run(
    ['kaggle', 'datasets', 'list', '--search', 'maize', '--max-size', '1'],
    capture_output=True, text=True
)
if 'Error' in result.stderr or '401' in result.stderr:
    print(f"❌ Auth failed: {result.stderr.strip()}")
    print("Check that you pasted the full token correctly (no spaces).")
else:
    print("✅ Kaggle authenticated! Proceed to Cell 4.")


## Cell 4 — 📥 Dataset Download, Extraction & Binary Labelling

**Datasets used:**
- **Dataset 1** (`hamishcrazeai/maize-in-field-dataset`) — 2,332 real-world field images, all diseased, no subfolders
- **Dataset 2** (`smaranjitghose/corn-or-maize-leaf-disease-dataset`) — 4,188 maize images across 4 class folders: `Blight`, `Common_Rust`, `Gray_Leaf_Spot`, `Healthy`

**Binary label mapping:** `Healthy` folder → 0, all disease folders → 1

In [ ]:
# ──────────────────────────────────────────────────────────────────────────
# CELL 4 — Dataset Download, Extraction & Binary Labelling
# Dataset 1: hamishcrazeai/maize-in-field-dataset      (field images, no folders)
# Dataset 2: smaranjitghose/corn-or-maize-leaf-disease-dataset (4 class folders)
# ──────────────────────────────────────────────────────────────────────────
import zipfile

IMG_EXTS = {'.jpg', '.jpeg', '.png', '.JPG', '.JPEG', '.PNG', '.bmp', '.BMP'}

def collect_images(directory, recursive=True):
    """Return all image file paths under a directory."""
    results = []
    if recursive:
        for root, _, files in os.walk(directory):
            for f in files:
                if os.path.splitext(f)[1] in IMG_EXTS:
                    results.append(os.path.join(root, f))
    else:
        for f in os.listdir(directory):
            if os.path.splitext(f)[1] in IMG_EXTS:
                results.append(os.path.join(directory, f))
    return results

d1_base = '/content/datasets/maize_in_field/'
d2_base = '/content/datasets/corn_disease/'

# ── DOWNLOAD (skips if already done this session) ─────────────────────────
if not os.path.exists(d1_base) or len(os.listdir(d1_base)) == 0:
    print("📥 Downloading Dataset 1: Maize in Field (hamishcrazeai)...")
    os.makedirs(d1_base, exist_ok=True)
    !kaggle datasets download -d hamishcrazeai/maize-in-field-dataset -p /content/datasets/ -q
    with zipfile.ZipFile('/content/datasets/maize-in-field-dataset.zip', 'r') as z:
        z.extractall(d1_base)
    os.remove('/content/datasets/maize-in-field-dataset.zip')
    print("   ✅ Done")
else:
    print("✅ Dataset 1 already present — skipping download")

if not os.path.exists(d2_base) or len(os.listdir(d2_base)) == 0:
    print("\n📥 Downloading Dataset 2: Corn/Maize Leaf Disease (smaranjitghose)...")
    os.makedirs(d2_base, exist_ok=True)
    !kaggle datasets download -d smaranjitghose/corn-or-maize-leaf-disease-dataset -p /content/datasets/ -q
    with zipfile.ZipFile('/content/datasets/corn-or-maize-leaf-disease-dataset.zip', 'r') as z:
        z.extractall(d2_base)
    os.remove('/content/datasets/corn-or-maize-leaf-disease-dataset.zip')
    print("   ✅ Done")
else:
    print("✅ Dataset 2 already present — skipping download")

# ── INSPECT EXTRACTED STRUCTURE ───────────────────────────────────────────
print("\n─── Dataset 1 top-level contents ────────────────────────")
for item in sorted(os.listdir(d1_base)):
    full = os.path.join(d1_base, item)
    tag  = "📁" if os.path.isdir(full) else "🖼️"
    print(f"  {tag} {item}")

print("\n─── Dataset 2 top-level contents ────────────────────────")
for item in sorted(os.listdir(d2_base)):
    full = os.path.join(d2_base, item)
    if os.path.isdir(full):
        n = len(collect_images(full))
        print(f"  📁 {item}/  ({n} images)")
    else:
        print(f"  📄 {item}")

# ── DATASET 1: field images → all DISEASED (no subfolders) ───────────────
print("\n─── Building Dataset 1 records ───────────────────────────")
d1_subdirs = [d for d in os.listdir(d1_base)
              if os.path.isdir(os.path.join(d1_base, d))]

dataset1_records = []
if d1_subdirs:
    # Subfolders exist — label by folder name
    for folder in sorted(d1_subdirs):
        label = 0 if ('healthy' in folder.lower() or 'normal' in folder.lower()) else 1
        imgs  = collect_images(os.path.join(d1_base, folder))
        dataset1_records += [(img, label) for img in imgs]
        print(f"  '{folder}' → {'Healthy' if label==0 else 'Diseased'} ({len(imgs)} images)")
else:
    # No subfolders → all images are diseased field images
    imgs = collect_images(d1_base, recursive=True)
    dataset1_records = [(img, 1) for img in imgs]
    print(f"  No subfolders — all {len(imgs)} images → DISEASED (label=1)")

# ── DATASET 2: walk all class folders, apply binary label ─────────────────
print("\n─── Building Dataset 2 records ───────────────────────────")
# This dataset is 100% maize — just walk all image-containing subfolders
# and map: 'healthy' folder → 0,  everything else → 1

dataset2_records = []

for root, dirs, files in os.walk(d2_base):
    imgs = [os.path.join(root, f) for f in files
            if os.path.splitext(f)[1] in IMG_EXTS]
    if not imgs:
        continue
    folder_name = os.path.basename(root)
    label = 0 if 'healthy' in folder_name.lower() else 1
    dataset2_records += [(img, label) for img in imgs]
    print(f"  '{folder_name}' → {'Healthy' if label==0 else 'Diseased'} ({len(imgs)} images)")

if len(dataset2_records) == 0:
    raise RuntimeError(
        "❌ No images found in Dataset 2. "
        "Check the extracted structure printed above."
    )

# ── COMBINE & SUMMARISE ───────────────────────────────────────────────────
all_records = dataset1_records + dataset2_records
all_paths   = [r[0] for r in all_records]
all_labels  = [r[1] for r in all_records]

total      = len(all_records)
n_healthy  = all_labels.count(0)
n_diseased = all_labels.count(1)

print(f"\n─── Combined dataset ─────────────────────────────────────")
print(f"  Dataset 1 (field images)  : {len(dataset1_records):,}  (all Diseased)")
print(f"  Dataset 2 (class folders) : {len(dataset2_records):,}")
print(f"  TOTAL                     : {total:,}")
print(f"  Healthy   (label=0)       : {n_healthy:,}  ({100*n_healthy/total:.1f}%)")
print(f"  Diseased  (label=1)       : {n_diseased:,}  ({100*n_diseased/total:.1f}%)")
print(f"  D:H ratio                 : {n_diseased/max(n_healthy,1):.2f}:1")

if n_healthy == 0 or n_diseased == 0:
    raise RuntimeError("❌ One class has zero images. Check folder names above.")

# ── STRATIFIED SPLIT (70 / 15 / 15) ──────────────────────────────────────
print("\n─── Stratified split (70 / 15 / 15) ─────────────────────")
labels_arr = np.array(all_labels)
sss1 = StratifiedShuffleSplit(1, test_size=0.15,      random_state=RANDOM_SEED)
sss2 = StratifiedShuffleSplit(1, test_size=0.15/0.85, random_state=RANDOM_SEED)

tv_idx, te_idx = next(sss1.split(all_paths, labels_arr))
tv_paths  = [all_paths[i]  for i in tv_idx]
tv_labels = labels_arr[tv_idx]
tr_idx, va_idx = next(sss2.split(tv_paths, tv_labels))

train_paths  = [tv_paths[i]  for i in tr_idx];  train_labels = tv_labels[tr_idx].tolist()
val_paths    = [tv_paths[i]  for i in va_idx];  val_labels   = tv_labels[va_idx].tolist()
test_paths   = [all_paths[i] for i in te_idx];  test_labels  = labels_arr[te_idx].tolist()

for name, paths, labels in [("Training",   train_paths, train_labels),
                              ("Validation", val_paths,   val_labels),
                              ("Test",       test_paths,  test_labels)]:
    h = labels.count(0); d = labels.count(1)
    print(f"  {name:12}: {len(paths):5,}  (H:{h:,}, D:{d:,})")

# ── CLASS WEIGHTS ─────────────────────────────────────────────────────────
cw = compute_class_weight('balanced', classes=np.array([0, 1]),
                          y=np.array(train_labels))
CLASS_WEIGHTS = {0: cw[0], 1: cw[1]}
print(f"\n  Class weights → Healthy: {CLASS_WEIGHTS[0]:.4f}, "
      f"Diseased: {CLASS_WEIGHTS[1]:.4f}")

# ── DATA GENERATORS ───────────────────────────────────────────────────────
train_df = pd.DataFrame({'filename': train_paths,
                         'class':    [str(l) for l in train_labels]})
val_df   = pd.DataFrame({'filename': val_paths,
                         'class':    [str(l) for l in val_labels]})
test_df  = pd.DataFrame({'filename': test_paths,
                         'class':    [str(l) for l in test_labels]})

train_gen_cfg = ImageDataGenerator(
    rescale=1./255, horizontal_flip=True,
    rotation_range=20, width_shift_range=0.10, height_shift_range=0.10,
    zoom_range=0.15, brightness_range=[0.80, 1.20], fill_mode='nearest'
)
eval_gen_cfg = ImageDataGenerator(rescale=1./255)

def make_gen(cfg, df, shuffle):
    return cfg.flow_from_dataframe(
        df, x_col='filename', y_col='class',
        target_size=(IMG_SIZE, IMG_SIZE), color_mode='rgb',
        class_mode='binary', batch_size=BATCH_SIZE,
        shuffle=shuffle, seed=RANDOM_SEED
    )

train_generator = make_gen(train_gen_cfg, train_df, shuffle=True)
val_generator   = make_gen(eval_gen_cfg,  val_df,   shuffle=False)
test_generator  = make_gen(eval_gen_cfg,  test_df,  shuffle=False)

print(f"\n  Class mapping : {train_generator.class_indices}")
print(f"  Train steps   : {len(train_generator)}")
print(f"  Val steps     : {len(val_generator)}")
print("\n✅ Data preparation complete!")


## Cell 5 — 🏗️ Build Model & Phase 1 Training (Feature Extraction)

**Phase 1**: The pre-trained base is **frozen**. Only the custom classification head is trained.  
Purpose: rapidly train the head before fine-tuning so the base isn't damaged by random gradients.

In [ ]:
# ──────────────────────────────────────────────────────────────────────────
# Architecture-specific configuration (thesis Table 3.8)
# ──────────────────────────────────────────────────────────────────────────
ARCH_CONFIG = {
    "MobileNetV2": {
        "builder":      MobileNetV2,
        "unfreeze_n":   30,          # Unfreeze top 30 layers in Phase 2
        "description":  "PRIMARY — 2.26M params | 0.30 GFLOPs | ~14 MB"
    },
    "Xception": {
        "builder":      Xception,
        "unfreeze_n":   40,          # Exit flow + 2 middle flow blocks
        "description":  "COMPARISON — 20.86M params | 8.40 GFLOPs | ~88 MB"
    },
    "InceptionV3": {
        "builder":      InceptionV3,
        "unfreeze_n":   93,          # ~30% of 311 layers (from mixed7 onward)
        "description":  "COMPARISON — 21.80M params | 5.70 GFLOPs | ~92 MB"
    },
    "VGG16": {
        "builder":      VGG16,
        "unfreeze_n":   4,           # block5 only (conv1-3 + pool)
        "description":  "BENCHMARK — 14.71M base params | 15.50 GFLOPs | ~59 MB base"
    },
    "ResNet50": {
        "builder":      ResNet50,
        "unfreeze_n":   53,          # ~30% of 177 layers (conv5 block)
        "description":  "BENCHMARK — 23.59M params | 4.10 GFLOPs | ~94 MB"
    }
}

cfg = ARCH_CONFIG[MODEL_NAME]
print(f"\n{'='*55}")
print(f"  ARCHITECTURE : {MODEL_NAME}")
print(f"  {cfg['description']}")
print(f"{'='*55}")

# ──────────────────────────────────────────────────────────────────────────
# Build model with custom binary head (thesis Table 3.7)
# Head: GAP → Dense(256,ReLU) → BatchNorm → Dropout(0.5) → Dense(1,sigmoid)
# ──────────────────────────────────────────────────────────────────────────
def build_model(model_name, img_size=224, dropout=0.5):
    builder    = ARCH_CONFIG[model_name]["builder"]
    base_model = builder(
        weights='imagenet',
        include_top=False,
        input_shape=(img_size, img_size, 3)
    )
    base_model.trainable = False   # FREEZE all base layers (Phase 1)

    inputs  = keras.Input(shape=(img_size, img_size, 3))
    x       = base_model(inputs, training=False)  # training=False keeps BN frozen
    x       = layers.GlobalAveragePooling2D()(x)  # Spatial maps → vector
    x       = layers.Dense(256, activation='relu')(x)
    x       = layers.BatchNormalization()(x)
    x       = layers.Dropout(dropout)(x)
    outputs = layers.Dense(1, activation='sigmoid')(x)  # P(Diseased) ∈ [0,1]

    model = keras.Model(inputs, outputs, name=f'{model_name}_binary')
    return model, base_model

model, base_model = build_model(MODEL_NAME, IMG_SIZE, DROPOUT_RATE)

print("\n─── Custom Classification Head ─────────────────────────────")
head_params = sum([tf.size(v).numpy() for v in model.trainable_variables])
total_params = model.count_params()
print(f"  Total parameters      : {total_params:,}")
print(f"  Head trainable params : {head_params:,}")
print(f"  Base frozen params    : {total_params - head_params:,}")
model.summary()

# ──────────────────────────────────────────────────────────────────────────
# Phase 1 — compile & train
# ──────────────────────────────────────────────────────────────────────────
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=PHASE1_LR),
    loss='binary_crossentropy',
    metrics=['accuracy',
             keras.metrics.Precision(name='precision'),
             keras.metrics.Recall(name='recall'),
             keras.metrics.AUC(name='auc')]
)

P1_CKPT = f'/content/{MODEL_NAME}_phase1_best.keras'
p1_callbacks = [
    EarlyStopping(
        monitor='val_loss', patience=5, min_delta=0.001,
        restore_best_weights=True, verbose=1
    ),
    ModelCheckpoint(P1_CKPT, monitor='val_loss', save_best_only=True, verbose=0)
]

print(f"\n{'='*55}")
print(f"  PHASE 1 — FEATURE EXTRACTION")
print(f"  Base: FROZEN  |  Head: TRAINING")
print(f"  LR: {PHASE1_LR}  |  Max epochs: {PHASE1_EPOCHS}")
print(f"  Early stop patience: 5")
print(f"{'='*55}\n")

history_p1 = model.fit(
    train_generator,
    epochs=PHASE1_EPOCHS,
    validation_data=val_generator,
    class_weight=CLASS_WEIGHTS,
    callbacks=p1_callbacks,
    verbose=1
)

best_p1_loss = min(history_p1.history['val_loss'])
best_p1_acc  = max(history_p1.history['val_accuracy'])
print(f"\n✅ Phase 1 done — best val_loss: {best_p1_loss:.4f}, best val_acc: {best_p1_acc*100:.2f}%")


## Cell 6 — 🔥 Phase 2 Fine-Tuning (max 20 epochs)

**Phase 2**: Selectively unfreeze the **top layers** of the base model.  
- Learning rate reduced to `1e-5` (10× smaller) to prevent destroying learned features  
- `ReduceLROnPlateau` halves LR if val_loss stalls for 5 epochs  
- Early stopping patience = 10 epochs  
- Model is **saved to Google Drive** at the end

In [ ]:
print(f"\n{'='*55}")
print(f"  PHASE 2 — SELECTIVE FINE-TUNING")
print(f"  LR: {PHASE2_LR}  |  Max epochs: {PHASE2_EPOCHS}")
print(f"  Early stop patience: 10  |  ReduceLROnPlateau patience: 5")
print(f"{'='*55}\n")

# ──────────────────────────────────────────────────────────────────────────
# Unfreeze top N layers of the base model (architecture-specific)
# ──────────────────────────────────────────────────────────────────────────
unfreeze_n          = ARCH_CONFIG[MODEL_NAME]["unfreeze_n"]
base_model.trainable = True                  # Un-lock the base first

total_base   = len(base_model.layers)
freeze_until = total_base - unfreeze_n       # Index from which to unfreeze

print(f"  Base model total layers : {total_base}")
print(f"  Unfreezing top {unfreeze_n} layers (index {freeze_until} → {total_base-1})")

for i, layer in enumerate(base_model.layers):
    # Layers below threshold: keep frozen
    # Layers at/above threshold: allow updates
    layer.trainable = (i >= freeze_until)

trainable_now = sum([tf.size(v).numpy() for v in model.trainable_variables])
print(f"  Trainable parameters now: {trainable_now:,}")

# ──────────────────────────────────────────────────────────────────────────
# Recompile with reduced learning rate
# ──────────────────────────────────────────────────────────────────────────
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=PHASE2_LR),
    loss='binary_crossentropy',
    metrics=['accuracy',
             keras.metrics.Precision(name='precision'),
             keras.metrics.Recall(name='recall'),
             keras.metrics.AUC(name='auc')]
)

# ──────────────────────────────────────────────────────────────────────────
# Phase 2 callbacks
# ──────────────────────────────────────────────────────────────────────────
P2_CKPT           = f'/content/{MODEL_NAME}_phase2_best.keras'
FINAL_MODEL_PATH  = os.path.join(DRIVE_OUTPUT, f'{MODEL_NAME}_final.keras')

p2_callbacks = [
    EarlyStopping(
        monitor='val_loss', patience=10, min_delta=0.001,
        restore_best_weights=True, verbose=1
    ),
    ModelCheckpoint(P2_CKPT, monitor='val_loss', save_best_only=True, verbose=0),
    ReduceLROnPlateau(
        monitor='val_loss', factor=0.5, patience=5,
        min_lr=1e-7, verbose=1
    )
]

# ──────────────────────────────────────────────────────────────────────────
# Phase 2 training  ★ MAX 20 EPOCHS ★
# ──────────────────────────────────────────────────────────────────────────
history_p2 = model.fit(
    train_generator,
    epochs=PHASE2_EPOCHS,     # ← Hard-capped at 20 for time constraint
    validation_data=val_generator,
    class_weight=CLASS_WEIGHTS,
    callbacks=p2_callbacks,
    verbose=1
)

# Save the final model to Google Drive immediately
print(f"\n💾 Saving model to Google Drive...")
model.save(FINAL_MODEL_PATH)
model.save(os.path.join(DRIVE_OUTPUT, f'{MODEL_NAME}_final.h5'))   # Also save .h5 backup
print(f"✅ Saved: {FINAL_MODEL_PATH}")

best_p2_loss = min(history_p2.history['val_loss'])
best_p2_acc  = max(history_p2.history['val_accuracy'])
p2_ran       = len(history_p2.history['loss'])
print(f"\n✅ Phase 2 complete ({p2_ran}/{PHASE2_EPOCHS} epochs ran)")
print(f"   Best val_loss : {best_p2_loss:.4f}")
print(f"   Best val_acc  : {best_p2_acc*100:.2f}%")


## Cell 7 — 📊 Evaluation & All Outputs

**This cell generates all 5 output artefacts** saved to your Google Drive:
1. 📈 Training history plot (Phase 1 + Phase 2 combined)  
2. 🟦 Confusion matrix  
3. 📉 ROC curve  
4. 📋 Metrics JSON (accuracy, sensitivity, specificity, F1, AUC-ROC, efficiency)  
5. 💾 Saved model (already done in Cell 6)

In [ ]:
# Load the best Phase 2 weights from checkpoint
model.load_weights(P2_CKPT)
print(f"✅ Best Phase 2 weights loaded")

# ──────────────────────────────────────────────────────────────────────────
# OUTPUT 1 — Training History Plot (Phase 1 + Phase 2 combined)
# ──────────────────────────────────────────────────────────────────────────
def plot_training_history(h1, h2, model_name, save_dir):
    acc      = h1.history['accuracy']      + h2.history['accuracy']
    val_acc  = h1.history['val_accuracy']  + h2.history['val_accuracy']
    loss     = h1.history['loss']          + h2.history['loss']
    val_loss = h1.history['val_loss']      + h2.history['val_loss']
    split    = len(h1.history['accuracy'])
    epochs   = range(1, len(acc) + 1)

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    fig.suptitle(f'{model_name} — Training History (Phase 1 + Phase 2)',
                 fontsize=14, fontweight='bold')

    for ax, train_vals, val_vals, ylabel in [
        (ax1, acc,  val_acc,  'Accuracy'),
        (ax2, loss, val_loss, 'Binary Cross-Entropy Loss')
    ]:
        ax.plot(epochs, train_vals, 'b-o', ms=3, label='Train')
        ax.plot(epochs, val_vals,   'r-o', ms=3, label='Validation')
        ax.axvline(x=split, color='grey', ls='--', lw=1.5,
                   label=f'Phase 1→2 (epoch {split})')
        ax.set_xlabel('Epoch', fontsize=11)
        ax.set_ylabel(ylabel, fontsize=11)
        ax.set_title(ylabel, fontsize=12)
        ax.legend(fontsize=10); ax.grid(True, alpha=0.3)

    ax1.set_ylim([max(0, min(acc+val_acc) - 0.05), 1.02])

    plt.tight_layout()
    out = os.path.join(save_dir, f'{model_name}_training_history.png')
    plt.savefig(out, dpi=150, bbox_inches='tight')
    plt.show()
    print(f"  ✅ OUTPUT 1 saved: {out}")

plot_training_history(history_p1, history_p2, MODEL_NAME, DRIVE_OUTPUT)

# ──────────────────────────────────────────────────────────────────────────
# PREDICTIONS on test set
# ──────────────────────────────────────────────────────────────────────────
print("\nRunning predictions on held-out test set...")
test_generator.reset()
y_prob  = model.predict(test_generator, verbose=1).flatten()
y_pred  = (y_prob >= 0.5).astype(int)
y_true  = test_generator.classes

# ── Metrics ───────────────────────────────────────────────────────────────
acc_s    = accuracy_score(y_true, y_pred)
prec_s   = precision_score(y_true, y_pred, zero_division=0)
rec_s    = recall_score(y_true, y_pred, zero_division=0)    # Sensitivity
f1_s     = f1_score(y_true, y_pred, zero_division=0)
auc_s    = roc_auc_score(y_true, y_prob)
cm       = confusion_matrix(y_true, y_pred)
TN, FP, FN, TP = cm.ravel()
spec_s   = TN / (TN + FP) if (TN + FP) > 0 else 0.0

# Wilson 95% CI for accuracy (thesis Section 3.12.3)
n        = len(y_true)
z        = 1.96
p_hat    = acc_s
denom    = 1 + z**2 / n
ci_mid   = p_hat + z**2 / (2*n)
ci_span  = z * np.sqrt(p_hat * (1 - p_hat) / n + z**2 / (4 * n**2))
ci_lo    = (ci_mid - ci_span) / denom
ci_hi    = (ci_mid + ci_span) / denom

# ──────────────────────────────────────────────────────────────────────────
# Print metrics table
# ──────────────────────────────────────────────────────────────────────────
print(f"\n{'='*55}")
print(f"  CLASSIFICATION PERFORMANCE — {MODEL_NAME}")
print(f"{'='*55}")
print(f"  Accuracy        : {acc_s*100:6.2f}%  (95% CI: {ci_lo*100:.2f}% – {ci_hi*100:.2f}%)")
print(f"  Precision       : {prec_s*100:6.2f}%")
print(f"  Sensitivity     : {rec_s*100:6.2f}%  (Recall — primary metric)")
print(f"  Specificity     : {spec_s*100:6.2f}%")
print(f"  F1 Score        : {f1_s*100:6.2f}%")
print(f"  AUC-ROC         : {auc_s:.4f}")
print(f"  ─────────────────────────────────────────")
print(f"  TP (correctly diseased)  : {TP:,}")
print(f"  TN (correctly healthy)   : {TN:,}")
print(f"  FP (healthy → diseased)  : {FP:,}  (false alarm)")
print(f"  FN (diseased → healthy)  : {FN:,}  ← most costly error!")
print(f"{'='*55}")

# ──────────────────────────────────────────────────────────────────────────
# OUTPUT 2 — Confusion Matrix
# ──────────────────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Healthy', 'Diseased'],
            yticklabels=['Healthy', 'Diseased'],
            ax=ax, linewidths=0.8, annot_kws={"size": 15, "weight": "bold"})
ax.set_xlabel('Predicted Label', fontsize=12)
ax.set_ylabel('True Label', fontsize=12)
ax.set_title(f'{MODEL_NAME} — Confusion Matrix (Test Set)', fontsize=13, fontweight='bold')
plt.tight_layout()
cm_path = os.path.join(DRIVE_OUTPUT, f'{MODEL_NAME}_confusion_matrix.png')
plt.savefig(cm_path, dpi=150, bbox_inches='tight')
plt.show()
print(f"  ✅ OUTPUT 2 saved: {cm_path}")

# ──────────────────────────────────────────────────────────────────────────
# OUTPUT 3 — ROC Curve
# ──────────────────────────────────────────────────────────────────────────
fpr, tpr, _ = roc_curve(y_true, y_prob)
fig, ax = plt.subplots(figsize=(7, 6))
ax.plot(fpr, tpr, 'b-', lw=2.5, label=f'{MODEL_NAME}  (AUC = {auc_s:.4f})')
ax.plot([0,1],[0,1], 'k--', lw=1, label='Random (AUC = 0.500)')
ax.fill_between(fpr, tpr, alpha=0.08, color='blue')
ax.set_xlabel('False Positive Rate  (1 − Specificity)', fontsize=12)
ax.set_ylabel('True Positive Rate  (Sensitivity)',      fontsize=12)
ax.set_title(f'{MODEL_NAME} — ROC Curve (Test Set)', fontsize=13, fontweight='bold')
ax.legend(loc='lower right', fontsize=11)
ax.grid(True, alpha=0.3)
ax.set_xlim([-0.01, 1.01]); ax.set_ylim([-0.01, 1.02])
plt.tight_layout()
roc_path = os.path.join(DRIVE_OUTPUT, f'{MODEL_NAME}_roc_curve.png')
plt.savefig(roc_path, dpi=150, bbox_inches='tight')
plt.show()
print(f"  ✅ OUTPUT 3 saved: {roc_path}")

# ──────────────────────────────────────────────────────────────────────────
# OUTPUT 4 — Computational Efficiency
# ──────────────────────────────────────────────────────────────────────────
print("\nMeasuring CPU inference time (50 images)...")
with tf.device('/CPU:0'):
    dummy = np.random.rand(1, IMG_SIZE, IMG_SIZE, 3).astype(np.float32)
    _ = model(dummy, training=False)    # warm-up
    times = []
    for _ in range(50):
        t0 = time.perf_counter()
        _ = model(dummy, training=False)
        times.append((time.perf_counter() - t0) * 1000)

inf_ms  = np.mean(times)
inf_std = np.std(times)
total_p = model.count_params()
train_p = sum([tf.size(v).numpy() for v in model.trainable_variables])
size_mb = os.path.getsize(os.path.join(DRIVE_OUTPUT, f'{MODEL_NAME}_final.h5')) / 1024**2

print(f"\n{'='*55}")
print(f"  COMPUTATIONAL EFFICIENCY — {MODEL_NAME}")
print(f"{'='*55}")
print(f"  Total parameters         : {total_p:,}")
print(f"  Trainable params (Ph2)   : {train_p:,}")
print(f"  Saved model (.h5)        : {size_mb:.2f} MB")
print(f"  CPU inference time       : {inf_ms:.2f} ± {inf_std:.2f} ms/image")
print(f"{'='*55}")

# ──────────────────────────────────────────────────────────────────────────
# OUTPUT 4 (cont.) — Save metrics JSON to Drive
# ──────────────────────────────────────────────────────────────────────────
results = {
    "architecture":              MODEL_NAME,
    "test_accuracy_pct":         round(acc_s  * 100, 2),
    "95ci_low_pct":              round(ci_lo  * 100, 2),
    "95ci_high_pct":             round(ci_hi  * 100, 2),
    "precision_pct":             round(prec_s * 100, 2),
    "sensitivity_recall_pct":    round(rec_s  * 100, 2),
    "specificity_pct":           round(spec_s * 100, 2),
    "f1_score_pct":              round(f1_s   * 100, 2),
    "auc_roc":                   round(auc_s, 4),
    "TP": int(TP), "TN": int(TN), "FP": int(FP), "FN": int(FN),
    "total_parameters":          int(total_p),
    "trainable_params_phase2":   int(train_p),
    "model_file_size_mb":        round(size_mb, 2),
    "cpu_inference_ms_mean":     round(inf_ms, 2),
    "cpu_inference_ms_std":      round(inf_std, 2),
    "phase1_epochs_ran":         len(history_p1.history['loss']),
    "phase2_epochs_ran":         len(history_p2.history['loss']),
    "best_val_loss_phase1":      round(min(history_p1.history['val_loss']), 4),
    "best_val_loss_phase2":      round(min(history_p2.history['val_loss']), 4),
    "train_samples":             len(train_paths),
    "val_samples":               len(val_paths),
    "test_samples":              len(test_paths),
    "class_weight_healthy":      round(CLASS_WEIGHTS[0], 4),
    "class_weight_diseased":     round(CLASS_WEIGHTS[1], 4),
}

json_path = os.path.join(DRIVE_OUTPUT, f'{MODEL_NAME}_results.json')
with open(json_path, 'w') as f:
    json.dump(results, f, indent=2)
print(f"  ✅ OUTPUT 4 saved: {json_path}")

# ──────────────────────────────────────────────────────────────────────────
# FINAL SUMMARY
# ──────────────────────────────────────────────────────────────────────────
print(f"\n{'='*55}")
print(f"  🏁  ALL DONE — {MODEL_NAME}")
print(f"{'='*55}")
print(f"  Accuracy     : {acc_s*100:.2f}%")
print(f"  Sensitivity  : {rec_s*100:.2f}%  ← key metric (miss no disease)")
print(f"  Specificity  : {spec_s*100:.2f}%")
print(f"  AUC-ROC      : {auc_s:.4f}")
print(f"  F1 Score     : {f1_s*100:.2f}%")
print(f"  Model Size   : {size_mb:.2f} MB")
print(f"  Inference    : {inf_ms:.2f} ms/image (CPU)")
print(f"{'─'*55}")
print(f"  Files in Google Drive  →  MyDrive/MaizeClassifier/{MODEL_NAME}/")
print(f"     ✓ {MODEL_NAME}_final.h5             (trained model)")
print(f"     ✓ {MODEL_NAME}_final.keras          (Keras format backup)")
print(f"     ✓ {MODEL_NAME}_training_history.png (learning curves)")
print(f"     ✓ {MODEL_NAME}_confusion_matrix.png (confusion matrix)")
print(f"     ✓ {MODEL_NAME}_roc_curve.png        (ROC curve)")
print(f"     ✓ {MODEL_NAME}_results.json         (all metrics)")
print(f"{'='*55}")
print("\n  Next step: go to your Google Drive and download all files.")
print("  Use the JSON file to fill Table 4.x in your thesis Chapter 4.")
